# Capstone #1: Research Agent

*Notebook 11*

Put it all together.

Combine web search, file search, and Code Interpreter into one agent.

It researches a topic, analyzes documents, and produces a structured report.


---

## 🔧 Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from openai import OpenAI
from agents import Agent, CodeInterpreterTool, FileSearchTool, Runner, ToolCallItem, WebSearchTool

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def called_tool_types(result) -> set[str]:
    """The set of hosted tool-call types the run actually invoked."""
    return {
        getattr(item.raw_item, "type", "")
        for item in result.new_items
        if isinstance(item, ToolCallItem)
    }


print(f"✅ Setup complete: Using {MODEL}")

---

## 🏗️ System Architecture

The research agent combines all three built-in tools from Lessons 08-10:

- **Web Search** (Lesson 08): finds current information and news

- **File Search** (Lesson 09): queries private reference documents

- **Code Interpreter** (Lesson 10): computes and analyzes quantitative data

**Workflow:** Topic → agent autonomously chooses tools → structured report.

---

## 📄 Phase 1: Build the Knowledge Base

We'll create an internal reference document and upload it to a vector store.

This gives the agent access to private context that web search won't find.

In [ ]:
# Internal reference document: private context the web doesn't have
reference_doc = """# AI Adoption Survey: Internal Research Summary

## Survey Overview
Conducted Q3 of this year. 847 respondents across technology, healthcare,
finance, and retail sectors. Companies ranged from 50 to 10,000+ employees.

## Key Findings

### Adoption Rates
- 73% of companies surveyed are actively using AI tools in production
- 18% are in pilot or evaluation phase
- 9% have no AI initiatives currently

### Primary Use Cases (ranked by adoption)
1. Customer support automation: 61%
2. Data analysis and reporting: 58%
3. Content generation: 47%
4. Code assistance: 44%
5. Document processing: 39%

### Biggest Barriers to Adoption
- Data privacy and security concerns: 67%
- Integration with existing systems: 54%
- Lack of skilled staff: 48%
- Cost and ROI uncertainty: 43%

### Budget Trends
- Average AI budget increase year-over-year: 34%
- Companies with dedicated AI teams: 41%
- Expected to increase AI investment next year: 78%

### Satisfaction
- Exceeded expectations: 29%
- Met expectations: 51%
- Below expectations: 20%
"""

# Save locally
doc_path = Path("ai_adoption_survey.txt")
doc_path.write_text(reference_doc)

# --------------------------------------------------------------
print(f"✅ Reference document created: {doc_path}")

#### Upload to Vector Store

In [ ]:
print("Uploading to vector store...\n")

vector_store = client.vector_stores.create(name="Research Agent Knowledge Base")

with open(doc_path, "rb") as file_handle:
    file_batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id=vector_store.id,
        files=[file_handle]
    )

if file_batch.file_counts.failed:
    raise RuntimeError(f"File processing failed: {file_batch.file_counts.failed} file(s)")

# --------------------------------------------------------------
print(f"✅ Vector store ready: {vector_store.id}")
print(f"   Files: {file_batch.file_counts.completed} processed")

With the private knowledge base ready, we can now build a single agent that has access to both public web data and our internal survey.

## 🤖 Phase 2: Build the Research Agent

All three tools in one agent: the instructions tell it exactly how to use each one.

Pattern to copy:

- List every tool the agent might need

- Give numbered step-by-step instructions for when to use each

- Start with default parameters

In your own project, you'd swap three things: the vector store contents, the agent instructions, and the `ResearchReport` fields.

In [ ]:
from pydantic import BaseModel
from typing import List

class ResearchReport(BaseModel):
    executive_summary: str
    key_findings: List[str]
    data_analysis: str
    sources: List[str]

research_instructions = (
    "You are a thorough research analyst. When given a research topic:\n\n"
    "1. Use web search to find current information, news, and trends\n"
    "2. Use file search to find relevant internal data and context\n"
    "3. Use the Python tool to compute statistics or analyze any numerical data\n"
    "4. Synthesize all findings into a structured report with these sections:\n"
    "    - Executive Summary (2-3 sentences)\n"
    "    - Key Findings (bullet points)\n"
    "    - Data Analysis (numbers and statistics)\n"
    "    - Sources\n\n"
    "Always cite your sources. Be specific with numbers."
)

research_agent = Agent(
    name="ResearchAgent",
    instructions=research_instructions,
    model=MODEL,
    output_type=ResearchReport,
    tools=[
        WebSearchTool(),
        FileSearchTool(
            vector_store_ids=[vector_store.id],
            max_num_results=3
        ),
        CodeInterpreterTool(tool_config={
            "type": "code_interpreter",
            # Auto mode creates a container or reuses one already in model context.
            "container": {"type": "auto"}
        })
    ]
)

# --------------------------------------------------------------
print("✅ Research agent ready")
print("    Tools: WebSearch + FileSearch + CodeInterpreter")

---

## 🚀 Phase 3: Run the Research Agent

The agent can use web search, file search, and Code Interpreter in one run.

⚠️ **Security note:** Tool output (web results, doc chunks) is untrusted.

Validate before passing it downstream (more in Lesson 21).

In [ ]:
research_topic = (
    "Research the current state of enterprise AI adoption: trends, barriers, and outlook. "
    "Use the internal survey and current web sources. "
    "Use the Python tool to estimate how many of the 847 respondents represent the 73% adoption rate."
)

print("🔬 RESEARCH AGENT RUNNING")
print("=" * 60)
print(f"Topic: {research_topic}")
print("\nResearching...\n")

# Multi-tool runs can fail at any step, so surface failures clearly
try:
    research_result = await Runner.run(research_agent, input=research_topic)
except Exception as error:
    print(f"Research failed: {error}")
    research_result = None

if research_result:
    report = research_result.final_output

    print("=" * 60)
    print("📋 RESEARCH REPORT")
    print("=" * 60)
    print(f"\nExecutive Summary:\n{report.executive_summary}\n")
    print("Key Findings:")
    for finding in report.key_findings:
        print(f"  • {finding}")
    print(f"\nData Analysis:\n{report.data_analysis}\n")
    print("Sources:")
    for source in report.sources:
        print(f"  • {source}")
    print("=" * 60)

    # Because we used output_type=ResearchReport, `report` is a validated Python object
    # with .executive_summary, .key_findings, etc. It has the expected shape for
    # downstream code, but facts and sources still need review.

    # Evidence: which of the three tools actually ran
    called = called_tool_types(research_result)
    expected_tools = {
        "Web Search": "web_search_call",
        "File Search": "file_search_call",
        "Code Interpreter": "code_interpreter_call",
    }
    print("\nTool use:")
    for label, call_type in expected_tools.items():
        print(f"  {label}: {'yes' if call_type in called else 'no'}")

    # Evidence: did the private survey fact (73%) reach the final report?
    report_json = report.model_dump_json()
    print(f"\nPrivate survey fact (73%) in report: {'yes' if '73%' in report_json else 'no'}")

## 🧹 Demo Cleanup

Removes the demo's sample files so the exercises start clean.

In [ ]:
# Clean up local sample files before exercises
for file in ["ai_adoption_survey.txt"]:
    path = Path(file)
    if path.exists():
        path.unlink()
        print(f"✅ Local file deleted: {file}")

---

## 💪 Practice Exercises

### Exercise 1: Export Research Report to Markdown

*Covers: saving agent output, writing results to a file*

Save the research report to a `.md` file using the existing `research_result`.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Export Research Report to Markdown
# --------------------------------------------------------------
# Objective: Save the existing research report to a markdown file.

from datetime import datetime

# TODO 1: Create a markdown header with the topic and current date
#          Hint: date_str = datetime.now().strftime("%Y-%m-%d")
#                header = f"# Research Report: {research_topic}\nDate: {date_str}\n\n"

# TODO 2: Build the report body from all four ResearchReport fields:
#          executive_summary, key_findings, data_analysis, sources
#          (research_result.final_output.executive_summary, and so on)

# TODO 3: Write the combined content to "report.md" using Path("report.md").write_text()

# TODO 4: Print a confirmation message showing the file path

# TODO 5: (Optional) Delete report.md when you're done to keep the folder clean

# --- Write your code below this line ---

---

### Exercise 2: Evaluate One Research Report

*Covers: judge-agent evaluation, scoring a structured report*

Apply the judge-agent evaluation pattern from Lesson 07 to one research report (focus on the judge step, not building a full test harness).

Good golden tests for this kind of agent check two things:

- **Content coverage**: did it include required facts?

- **Tool-backed grounding**: did it cite sources or use internal data when expected?

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Evaluate One Research Report
# --------------------------------------------------------------
# Objective: Score one research report using a judge agent (Lesson 07 pattern).

from pydantic import BaseModel, Field

# A golden test: the topic we already ran, with criteria the report should meet
golden_test = {
    "topic": "Current state of AI adoption in enterprise",
    "must_contain": ["73%", "privacy"],  # facts from the internal survey
}

# Define an EvalResult model for the judge agent
class EvalResult(BaseModel):
    passed: bool
    score: int = Field(ge=0, le=10)
    feedback: str

# TODO 1: Create a judge agent using REASONING_MODEL and output_type=EvalResult
#          Instructions: score the report against the golden_test criteria

# TODO 2: Build the judge input from golden_test and
#          report.model_dump_json(). Judge the captured report.
#          Do not rerun the research agent.

# TODO 3: Check tool use deterministically:
#          required = {"web_search_call", "file_search_call", "code_interpreter_call"}
#          Confirm required <= called_tool_types(research_result)

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**One agent can choose among multiple tools:**

- Web search for current information

- File search for private context

- Code Interpreter for computation

- Inspect run items to prove which tools actually ran
<br>
<br>

**Instructions guide tool use:**

- Explicit instructions about when to use each tool produce more consistent results

- Structured output (`output_type=ResearchReport`) guarantees the report's shape on a successful run, not its content

- Handle errors where the caller can recover: a multi-tool run can fail at any step
<br>
<br>

**Files and vector stores are separate resources:**

- Delete uploaded files with `client.files.delete(...)` and the store with `client.vector_stores.delete(...)`

- In a real app, keep the store and reuse its ID until the documents change

---

## 📍 Next Step

**Notebook 12: Handoffs**  

Learn how to route tasks between specialized agents using the handoff pattern.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-11-capstone-1--research-agent)

---

#### Cleanup: Delete Uploaded Files and Vector Store

Uploaded files and vector stores persist on OpenAI's servers until deleted.

Deleting the store does not delete the files you uploaded, so remove both.

Run this cell when you're done to avoid ongoing storage charges.

In [ ]:
# Vector stores and uploaded Files are separate objects: delete both
uploaded_file_ids = [
    store_file.id
    for store_file in client.vector_stores.files.list(
        vector_store_id=vector_store.id
    ).data
]

for file_id in uploaded_file_ids:
    client.files.delete(file_id)

client.vector_stores.delete(vector_store.id)

print(f"✅ Uploaded files deleted: {len(uploaded_file_ids)}")
print(f"✅ Vector store deleted: {vector_store.id}")

---